# CAMUS Echocardiography Segmentation in Google Colab

This notebook clones the GitHub repo, mounts Google Drive, installs Colab-safe dependencies, runs tests, trains a model, evaluates it, and saves outputs back to Drive.

## 1. Use a GPU Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU** before running the training cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/yasser2652/Deep-Echocardiographic-segmentation-.git'
PROJECT_DIR = '/content/DeepEchoSeg'

!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

In [ ]:
# Colab usually already has CUDA-enabled torch installed. Avoid reinstalling torch unless you know you need to.
!pip install -q SimpleITK nibabel albumentations pyyaml tqdm pandas scipy scikit-image matplotlib pytest

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configure CAMUS Paths

Put CAMUS in Google Drive, usually at `/content/drive/MyDrive/CAMUS`. If you uploaded `CAMUS.zip`, uncomment the unzip cell below.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/CAMUS'
OUTPUT_DIR = '/content/drive/MyDrive/camus_outputs'
RUN_NAME = 'baseline_unet_colab'

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Optional: unzip CAMUS.zip from Drive if needed.
# CAMUS_ZIP = '/content/drive/MyDrive/CAMUS.zip'
# !unzip -q "$CAMUS_ZIP" -d /content/drive/MyDrive

## 3. Run Tests

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 pytest -q tests

## 4. Optional Smoke Training Run

Run this first to verify the training loop in Colab without CAMUS.

In [ ]:
!python -m src.train --config config.yaml --create-dummy-data --epochs 1 --batch-size 2 --image-size 64 --model baseline_unet --output-dir "$OUTPUT_DIR" --run-name smoke_colab --device "$DEVICE"

## 5. Train on CAMUS

Start with a small epoch count, then increase after confirming paths and GPU memory.

In [ ]:
EPOCHS = 100
BATCH_SIZE = 8
IMAGE_SIZE = 256
MODEL = 'baseline_unet'

!python -m src.train \
  --config config.yaml \
  --data-root "$DATA_ROOT" \
  --output-dir "$OUTPUT_DIR" \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --epochs "$EPOCHS" \
  --batch-size "$BATCH_SIZE" \
  --image-size "$IMAGE_SIZE" \
  --device "$DEVICE" \
  --mixed-precision

## 6. Evaluate

In [ ]:
CHECKPOINT = f'{OUTPUT_DIR}/{RUN_NAME}/best.pth'
EVAL_DIR = f'{OUTPUT_DIR}/{RUN_NAME}/evaluation'

!python -m src.evaluate \
  --checkpoint "$CHECKPOINT" \
  --data-root "$DATA_ROOT" \
  --output-dir "$EVAL_DIR" \
  --split test \
  --device "$DEVICE"

## 7. Predict a Patient Folder

In [ ]:
# Change this to an actual patient folder after CAMUS is mounted.
PATIENT_DIR = f'{DATA_ROOT}/patient0001'
PRED_DIR = f'{OUTPUT_DIR}/{RUN_NAME}/predictions_patient0001'

!python -m src.predict \
  --checkpoint "$CHECKPOINT" \
  --input "$PATIENT_DIR" \
  --output-dir "$PRED_DIR" \
  --device "$DEVICE" \
  --save-confidence \
  --postprocess